# **MDlens**
### Multi-Complex Comparative Molecular Dynamics Analysis

MDlens is a Google Colab-based Python notebook for comparative analysis of multiple protein–ligand MD simulations. It handles multiple complexes simultaneously, supports multiple replicates per complex (with automatic averaging and error-band visualization), and produces publication-ready plots alongside quantitative tables.


###**Author:** Omar Arias-Gaguancela, PhD | SciLearningWorkshops LLC  
###**Version:** 1.0  
---

In [ ]:
#@title ### Cell 1 — Install Required Packages
#@markdown Run this cell first to install all dependencies.

import subprocess, sys

packages = [
    'MDAnalysis',
    'matplotlib',
    'seaborn',
    'numpy',
    'scipy',
    'pandas',
    'scikit-learn',
    'ipywidgets',
    'tqdm',
]

print('Installing MDlens dependencies...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

print('\nAll packages installed. Proceed to Cell 2.')

In [ ]:
#@title ### Cell 2 — Mount Google Drive & Set Working Directory
#@markdown Mount your Google Drive and navigate to the folder containing your MD files.
#@markdown
#@markdown **Tip:** Use `/` notation to navigate subfolders (e.g., `MyProject/Simulations`).

import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive'
    print(f'Google Drive mounted at /content/drive/MyDrive')
except ImportError:
    DRIVE_ROOT = os.path.expanduser('~')
    print(f'Not running in Colab. Using home directory: {DRIVE_ROOT}')

# --- Interactive Folder Search Widget ---
search_input = widgets.Text(
    placeholder='Type folder name to search...',
    description='Search:',
    layout=widgets.Layout(width='400px')
)
search_button = widgets.Button(
    description='Search',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)
folder_dropdown = widgets.Dropdown(
    options=[],
    description='Folder:',
    layout=widgets.Layout(width='600px')
)
set_dir_button = widgets.Button(
    description='Set Directory',
    button_style='success',
    layout=widgets.Layout(width='150px')
)
status_out = widgets.Output()

def search_folders(b):
    query = search_input.value.strip()
    if not query:
        return
    matches = []
    for root, dirs, files in os.walk(DRIVE_ROOT):
        for d in dirs:
            if query.lower() in d.lower():
                full = os.path.join(root, d)
                display_path = full.replace(DRIVE_ROOT + '/', '')
                matches.append((display_path, full))
        if len(matches) >= 50:
            break
    if matches:
        folder_dropdown.options = [(m[0], m[1]) for m in matches]
        with status_out:
            clear_output()
            print(f'Found {len(matches)} matching folder(s). Select one and click Set Directory.')
    else:
        with status_out:
            clear_output()
            print('No folders found matching that query.')

def set_directory(b):
    if folder_dropdown.value:
        os.chdir(folder_dropdown.value)
        with status_out:
            clear_output()
            print(f'Working directory set to:\n  {os.getcwd()}')
            print(f'\nFiles in directory:')
            for f in sorted(os.listdir('.'))[:30]:
                print(f'  {f}')

search_button.on_click(search_folders)
set_dir_button.on_click(set_directory)

print('\n--- Google Drive Folder Navigator ---')
print('Tip: Use "/" in search to navigate subfolders (e.g., "Project/Simulations")')
display(widgets.HBox([search_input, search_button]))
display(folder_dropdown)
display(set_dir_button)
display(status_out)

In [ ]:
#@title ### Cell 3 — MDComplex and MDComparator Class Definitions
#@markdown Core class definitions. Run once before using any other cell.

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import seaborn as sns
from scipy.ndimage import gaussian_filter
from scipy.interpolate import interp1d
from scipy.spatial.distance import cdist
from scipy.stats import chi2
from sklearn.decomposition import PCA
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
from MDAnalysis.analysis.base import AnalysisBase
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
matplotlib.rcParams['figure.dpi'] = 100

# ─────────────────────────────────────────────────────────────────────────────
# Utility helpers
# ─────────────────────────────────────────────────────────────────────────────

def _safe_name(name):
    """Convert complex name to a filesystem-safe string."""
    return name.replace(' ', '_').replace('/', '-').replace('\\', '-')


def _interpolate_to_length(arr, target_len):
    """Linearly interpolate 1D array to target_len points."""
    src_x = np.linspace(0, 1, len(arr))
    tgt_x = np.linspace(0, 1, target_len)
    return interp1d(src_x, arr, kind='linear')(tgt_x)


def _align_arrays(arrays):
    """
    Given a list of 1D arrays of potentially different lengths,
    interpolate all to the length of the longest, then return
    a 2D array (n_replicates x max_len).
    """
    max_len = max(len(a) for a in arrays)
    aligned = np.array([_interpolate_to_length(a, max_len) for a in arrays])
    return aligned


def _align_2d_arrays(arrays):
    """
    Given a list of 2D arrays (residues x frames) of potentially different
    frame counts, interpolate frame axis to max length.
    Returns 3D array (n_rep x n_res x max_frames).
    """
    n_res = arrays[0].shape[0]
    max_frames = max(a.shape[1] for a in arrays)
    aligned = np.zeros((len(arrays), n_res, max_frames))
    for i, a in enumerate(arrays):
        for r in range(n_res):
            aligned[i, r] = _interpolate_to_length(a[r], max_frames)
    return aligned


def _confidence_ellipse(x, y, ax, confidence=0.95, **kwargs):
    """Draw a confidence ellipse for 2D data on ax."""
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    scale = np.sqrt(chi2.ppf(confidence, df=2))
    w, h = 2 * scale * np.sqrt(vals)
    ellipse = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=w, height=h,
        angle=angle,
        **kwargs
    )
    ax.add_patch(ellipse)


# ─────────────────────────────────────────────────────────────────────────────
# MDComplex
# ─────────────────────────────────────────────────────────────────────────────

class MDComplex:
    """
    Represents one protein-ligand MD system with one or more replicates.

    Parameters
    ----------
    name : str
        Human-readable label used in plot legends and output file names.
    topology : str
        Path to topology file (PDB, GRO, PSF, or any MDAnalysis-supported format).
    trajectories : str or list of str
        One trajectory file or a list of replicate trajectories
        (XTC, TRR, DCD, NetCDF, or any MDAnalysis-supported format).
    protein_selection : str
        MDAnalysis selection string for protein atoms. Default: 'protein'.
    ligand_selection : str
        MDAnalysis selection string for ligand atoms. Default: 'resname LIG'.
    total_time_ns : float or None
        Total simulation time in nanoseconds (e.g. 100 for a 100 ns run).
        When set, time axes are built as linspace(0, total_time_ns, n_frames),
        which is reliable for all trajectory formats including DCD.
    resid_offset : int
        Residue number offset applied when matching residues across complexes.
    """

    def __init__(
        self,
        name,
        topology,
        trajectories,
        protein_selection='protein',
        ligand_selection='resname LIG',
        total_time_ns=None,
        resid_offset=0,
    ):
        self.name = name
        self.topology = topology
        self.trajectories = [trajectories] if isinstance(trajectories, str) else list(trajectories)
        self.protein_selection = protein_selection
        self.ligand_selection = ligand_selection
        self.total_time_ns = total_time_ns
        self.resid_offset = resid_offset

        # Results cache — populated by compute()
        self._results = {}
        self._n_residues = None
        self._residue_ids = None
        self._residue_names = None
        self._computed = False

        # Validate files exist
        self._validate_files()

    # ------------------------------------------------------------------
    # File validation
    # ------------------------------------------------------------------

    def _validate_files(self):
        if not os.path.isfile(self.topology):
            raise FileNotFoundError(f'Topology not found: {self.topology}')
        for t in self.trajectories:
            if not os.path.isfile(t):
                raise FileNotFoundError(f'Trajectory not found: {t}')

    # ------------------------------------------------------------------
    # Universe helpers
    # ------------------------------------------------------------------

    def _make_universe(self, traj):
        return mda.Universe(self.topology, traj)

    def _get_time_axis(self, u):
        """Build time axis in ns. Uses total_time_ns when set (reliable);
        falls back to ts.time which is unreliable for DCD/CHARMM files."""
        n_frames = len(u.trajectory)
        if self.total_time_ns is not None:
            return np.linspace(0, self.total_time_ns, n_frames)
        # Fallback: read from trajectory (may be wrong for DCD)
        times = []
        for ts in u.trajectory:
            times.append(ts.time)
        return np.array(times) / 1000.0  # ps → ns

    # ------------------------------------------------------------------
    # Per-replicate metric computation
    # ------------------------------------------------------------------

    def _compute_rmsd(self, u):
        """Backbone RMSD vs time, reference = frame 0."""
        ref = self._make_universe(self.trajectories[0])
        ref.trajectory[0]
        backbone = u.select_atoms(self.protein_selection + ' and backbone')
        if len(backbone) == 0:
            backbone = u.select_atoms(self.protein_selection + ' and name CA')
        ref_backbone = ref.select_atoms(self.protein_selection + ' and backbone')
        if len(ref_backbone) == 0:
            ref_backbone = ref.select_atoms(self.protein_selection + ' and name CA')
        r = rms.RMSD(backbone, ref_backbone, superposition=True)
        r.run()
        return r.results.rmsd[:, 2]

    def _compute_ligand_rmsd(self, u):
        """Ligand heavy-atom RMSD: align protein backbone first, then measure ligand."""
        ref = self._make_universe(self.trajectories[0])
        ref.trajectory[0]
        ligand = u.select_atoms(self.ligand_selection + ' and not name H*')
        if len(ligand) == 0:
            print(f'  ⚠  [{self.name}] No ligand atoms found with selection: {self.ligand_selection}')
            return np.zeros(len(u.trajectory))
        backbone = u.select_atoms(self.protein_selection + ' and backbone')
        if len(backbone) == 0:
            backbone = u.select_atoms(self.protein_selection + ' and name CA')
        ref_backbone = ref.select_atoms(self.protein_selection + ' and backbone')
        if len(ref_backbone) == 0:
            ref_backbone = ref.select_atoms(self.protein_selection + ' and name CA')
        ref_ligand = ref.select_atoms(self.ligand_selection + ' and not name H*')
        rmsd_vals = []
        ref.trajectory[0]
        ref_lig_pos = ref_ligand.positions.copy()
        ref_bb_pos = ref_backbone.positions.copy()
        for ts in u.trajectory:
            # Align backbone
            R, t = align.rotation_matrix(backbone.positions, ref_bb_pos)
            lig_pos = ligand.positions - backbone.center_of_mass()
            lig_pos = lig_pos @ R.T + ref_backbone.center_of_mass()
            diff = lig_pos - ref_lig_pos
            rmsd_vals.append(np.sqrt((diff ** 2).sum(axis=1).mean()))
        return np.array(rmsd_vals)

    def _compute_rg(self, u):
        """Radius of gyration of the protein vs time (Å)."""
        protein = u.select_atoms(self.protein_selection)
        rg_vals = []
        for ts in u.trajectory:
            rg_vals.append(protein.radius_of_gyration())
        return np.array(rg_vals)

    def _compute_rmsf(self, u):
        """Per-residue Cα RMSF, aligned to frame 0."""
        ca = u.select_atoms(self.protein_selection + ' and name CA')
        if len(ca) == 0:
            ca = u.select_atoms(self.protein_selection)
        aligner = align.AlignTraj(u, u, select=self.protein_selection + ' and name CA',
                                  in_memory=True)
        aligner.run()
        r = rms.RMSF(ca)
        r.run()
        return r.results.rmsf

    def _compute_hbonds(self, u):
        """
        Protein-ligand H-bond count vs time.
        NOTE: Non-standard ligands may require explicit donor/acceptor definitions.
        """
        try:
            hba = HydrogenBondAnalysis(
                u,
                donors_sel=self.protein_selection,
                acceptors_sel=self.ligand_selection,
            )
            hba.run()
            counts_da = hba.count_by_time()

            hba2 = HydrogenBondAnalysis(
                u,
                donors_sel=self.ligand_selection,
                acceptors_sel=self.protein_selection,
            )
            hba2.run()
            counts_ad = hba2.count_by_time()

            n = min(len(counts_da), len(counts_ad))
            return counts_da[:n] + counts_ad[:n]
        except Exception as e:
            print(f'  ⚠  [{self.name}] H-bond analysis failed: {e}')
            print(f'     Tip: Non-standard ligands may require explicit donor/acceptor hints.')
            return np.zeros(len(u.trajectory))

    def _compute_pl_distance(self, u):
        """Minimum protein-ligand distance vs time (Å)."""
        protein = u.select_atoms(self.protein_selection)
        ligand = u.select_atoms(self.ligand_selection)
        if len(ligand) == 0:
            return np.zeros(len(u.trajectory))
        dists = []
        for ts in u.trajectory:
            d = cdist(protein.positions, ligand.positions)
            dists.append(d.min())
        return np.array(dists)

    def _compute_binding_energy(self, u):
        """
        Simplified MM binding energy: Coulomb + Lennard-Jones between
        protein and ligand atoms (kJ/mol, unitless atom-type approximation).
        Uses MDAnalysis inter-group energies if available, else estimates
        from pairwise distances.
        """
        try:
            from MDAnalysis.analysis.energy import EnergyReader  # noqa: F401
            # If GROMACS EDR file is available, use it (not implemented here;
            # falls through to distance-based estimate)
            raise ImportError
        except Exception:
            pass

        protein = u.select_atoms(self.protein_selection)
        ligand = u.select_atoms(self.ligand_selection)
        if len(ligand) == 0:
            return np.zeros(len(u.trajectory))

        # Generic charges: assume +0.2 for protein polar atoms, -0.2 for ligand
        # and default LJ epsilon=0.1 kJ/mol, sigma=3.5 Å (approximate C–C)
        epsilon = 0.1       # kJ/mol
        sigma = 3.5         # Å
        q_p = 0.2           # elementary charge proxy
        q_l = -0.2
        k_e = 138.935       # kJ·Å/(mol·e²)

        energies = []
        for ts in u.trajectory:
            r = cdist(protein.positions, ligand.positions)  # Å
            # Only include pairs within 12 Å cutoff
            mask = r < 12.0
            r_cut = r[mask]
            if len(r_cut) == 0:
                energies.append(0.0)
                continue
            r_cut = np.clip(r_cut, 1.0, None)  # avoid singularity
            lj = 4 * epsilon * ((sigma / r_cut) ** 12 - (sigma / r_cut) ** 6)
            coul = k_e * q_p * q_l / r_cut
            energies.append((lj + coul).sum())
        return np.array(energies)

    def _compute_segmented_rmsf(self, u, n_segments=10):
        """
        Split trajectory into n_segments equal segments and compute
        per-residue RMSF within each segment.
        Returns 2D array: shape (n_residues, n_segments).
        Inspired by eRMSF (Arantes et al., JCIM 2025).
        """
        ca = u.select_atoms(self.protein_selection + ' and name CA')
        if len(ca) == 0:
            ca = u.select_atoms(self.protein_selection)
        n_frames = len(u.trajectory)
        seg_size = n_frames // n_segments
        if seg_size < 2:
            print(f'  ⚠  [{self.name}] Too few frames for {n_segments} segments. Using {max(1, n_frames//2)} segments.')
            n_segments = max(1, n_frames // 2)
            seg_size = n_frames // n_segments

        seg_rmsf = np.zeros((len(ca), n_segments))
        for seg_idx in range(n_segments):
            start = seg_idx * seg_size
            stop = start + seg_size if seg_idx < n_segments - 1 else n_frames
            positions = []
            for ts in u.trajectory[start:stop]:
                positions.append(ca.positions.copy())
            positions = np.array(positions)  # (frames, n_ca, 3)
            mean_pos = positions.mean(axis=0)
            diff = positions - mean_pos
            seg_rmsf[:, seg_idx] = np.sqrt((diff ** 2).sum(axis=2).mean(axis=0))
        return seg_rmsf

    def _compute_contact_residues(self, u, cutoff=5.0, min_freq=0.30):
        """
        Identify protein residues within cutoff Å of the ligand.
        Returns dict: {resid: (contact_freq, mean_min_dist, resname)}
        """
        ligand = u.select_atoms(self.ligand_selection)
        if len(ligand) == 0:
            return {}
        protein = u.select_atoms(self.protein_selection)
        residues = protein.residues
        n_frames = len(u.trajectory)
        contact_counts = np.zeros(len(residues))
        dist_sums = np.zeros(len(residues))

        for ts in u.trajectory:
            lig_pos = ligand.positions
            for ri, res in enumerate(residues):
                res_pos = res.atoms.positions
                d = cdist(res_pos, lig_pos).min()
                dist_sums[ri] += d
                if d <= cutoff:
                    contact_counts[ri] += 1

        result = {}
        for ri, res in enumerate(residues):
            freq = contact_counts[ri] / n_frames
            if freq >= min_freq:
                rid = int(res.resid) + self.resid_offset
                result[rid] = (freq, dist_sums[ri] / n_frames, res.resname)
        return result

    def _compute_fel_data(self, universes):
        """
        Compute FEL using RMSD and Rg as collective variables.
        For multiple replicates, concatenate all trajectories.
        Returns (rmsd_all, rg_all, time_all, frame_universe_idx_pairs)
        """
        rmsd_all, rg_all, time_all = [], [], []
        frame_map = []  # (universe_index, local_frame_index)

        for ui, u in enumerate(universes):
            ref = self._make_universe(self.trajectories[0])
            ref.trajectory[0]
            backbone = u.select_atoms(self.protein_selection + ' and backbone')
            if len(backbone) == 0:
                backbone = u.select_atoms(self.protein_selection + ' and name CA')
            ref_bb = ref.select_atoms(self.protein_selection + ' and backbone')
            if len(ref_bb) == 0:
                ref_bb = ref.select_atoms(self.protein_selection + ' and name CA')
            rmsd_obj = rms.RMSD(backbone, ref_bb, superposition=True)
            rmsd_obj.run()
            rmsd_rep = rmsd_obj.results.rmsd[:, 2]

            protein = u.select_atoms(self.protein_selection)
            # Use _get_time_axis so total_time_ns is always respected.
            # ts.time is unreliable for DCD/CHARMM: MDAnalysis reads the
            # integration timestep from the header, not the output frequency.
            time_rep = self._get_time_axis(u)  # ns array, length = n_frames
            rg_rep = []
            for ti, ts in enumerate(u.trajectory):
                rg_rep.append(protein.radius_of_gyration())
                frame_map.append((ui, ti))
            rmsd_all.extend(rmsd_rep.tolist())
            rg_all.extend(rg_rep)
            time_all.extend(time_rep.tolist())

        return (np.array(rmsd_all), np.array(rg_all),
                np.array(time_all), frame_map)

    # ------------------------------------------------------------------
    # Main compute entry point
    # ------------------------------------------------------------------

    def compute(self, n_segments=10):
        """
        Load all replicates and compute all metrics. Results are stored
        in self._results and accessed by MDComparator.
        """
        print(f'\n━━━ Computing metrics for [{self.name}] ━━━')
        n_rep = len(self.trajectories)

        # ── Build universes ──────────────────────────────────────────
        universes = []
        for i, traj in enumerate(self.trajectories):
            print(f'  Loading replicate {i+1}/{n_rep}: {os.path.basename(traj)}')
            universes.append(self._make_universe(traj))

        # Residue metadata from first universe
        u0 = universes[0]
        ca0 = u0.select_atoms(self.protein_selection + ' and name CA')
        self._n_residues = len(ca0)
        self._residue_ids = np.array([r.resid + self.resid_offset for r in ca0.residues])
        self._residue_names = np.array([r.resname for r in ca0.residues])

        # ── Time axis ───────────────────────────────────────────────
        time_arrays = [self._get_time_axis(u) for u in universes]
        max_len = max(len(t) for t in time_arrays)
        time_interp = np.linspace(time_arrays[0][0], time_arrays[0][-1], max_len)

        # Helper: run a metric on each replicate and aggregate
        def _run_metric(fn_name, fn, universes):
            try:
                print(f'  Computing {fn_name}...')
                arrays = [fn(u) for u in universes]
                if n_rep == 1:
                    return arrays[0], None
                aligned = _align_arrays(arrays)
                return aligned.mean(axis=0), aligned.std(axis=0)
            except Exception as e:
                print(f'  ✗ {fn_name} failed: {e}')
                return None, None

        # ── RMSD ────────────────────────────────────────────────────
        rmsd_mean, rmsd_std = _run_metric('RMSD', self._compute_rmsd, universes)
        self._results['rmsd'] = (time_interp if rmsd_mean is not None else None,
                                 rmsd_mean, rmsd_std)

        # ── Ligand RMSD ─────────────────────────────────────────────
        lig_rmsd_mean, lig_rmsd_std = _run_metric('Ligand RMSD', self._compute_ligand_rmsd, universes)
        self._results['ligand_rmsd'] = (time_interp, lig_rmsd_mean, lig_rmsd_std)

        # ── Rg ──────────────────────────────────────────────────────
        rg_mean, rg_std = _run_metric('Rg', self._compute_rg, universes)
        self._results['rg'] = (time_interp, rg_mean, rg_std)

        # ── RMSF ────────────────────────────────────────────────────
        try:
            print('  Computing RMSF...')
            rmsf_arrays = [self._compute_rmsf(u) for u in universes]
            if n_rep == 1:
                rmsf_mean, rmsf_std = rmsf_arrays[0], None
            else:
                rmsf_stack = np.array(rmsf_arrays)
                rmsf_mean = rmsf_stack.mean(axis=0)
                rmsf_std = rmsf_stack.std(axis=0)
        except Exception as e:
            print(f'  ✗ RMSF failed: {e}')
            rmsf_mean, rmsf_std = None, None
        self._results['rmsf'] = (self._residue_ids, rmsf_mean, rmsf_std)

        # ── H-bonds ─────────────────────────────────────────────────
        hb_mean, hb_std = _run_metric('H-bonds', self._compute_hbonds, universes)
        self._results['hbonds'] = (time_interp, hb_mean, hb_std)

        # ── P-L distance ─────────────────────────────────────────────
        pld_mean, pld_std = _run_metric('P-L Distance', self._compute_pl_distance, universes)
        self._results['pl_distance'] = (time_interp, pld_mean, pld_std)

        # ── Binding energy ──────────────────────────────────────────
        be_mean, be_std = _run_metric('Binding Energy', self._compute_binding_energy, universes)
        self._results['binding_energy'] = (time_interp, be_mean, be_std)

        # ── Segmented RMSF heatmap ────────────────────────────────
        try:
            print(f'  Computing segmented RMSF ({n_segments} segments)...')
            seg_arrays = [self._compute_segmented_rmsf(u, n_segments) for u in universes]
            if n_rep == 1:
                seg_mean = seg_arrays[0]
            else:
                seg_aligned = _align_2d_arrays(seg_arrays)
                seg_mean = seg_aligned.mean(axis=0)
        except Exception as e:
            print(f'  ✗ Segmented RMSF failed: {e}')
            seg_mean = None
        self._results['seg_rmsf'] = seg_mean

        # ── Contact residues ─────────────────────────────────────────
        try:
            print('  Computing contact residues...')
            contact_all = [self._compute_contact_residues(u) for u in universes]
            if n_rep == 1:
                contact_merged = contact_all[0]
            else:
                # Merge: average freq and mean_dist across replicates
                all_resids = set()
                for c in contact_all:
                    all_resids.update(c.keys())
                contact_merged = {}
                for rid in all_resids:
                    freqs = [c[rid][0] for c in contact_all if rid in c]
                    dists = [c[rid][1] for c in contact_all if rid in c]
                    resname = next(c[rid][2] for c in contact_all if rid in c)
                    contact_merged[rid] = (np.mean(freqs), np.mean(dists), resname)
        except Exception as e:
            print(f'  ✗ Contact residue detection failed: {e}')
            contact_merged = {}
        self._results['contact_residues'] = contact_merged

        # ── FEL data ─────────────────────────────────────────────────
        try:
            print('  Computing FEL data...')
            fel_data = self._compute_fel_data(universes)
        except Exception as e:
            print(f'  ✗ FEL data failed: {e}')
            fel_data = None
        self._results['fel_data'] = fel_data

        self._computed = True
        print(f'  ✓ Done. [{self.name}]')
        return self


# ─────────────────────────────────────────────────────────────────────────────
# MDComparator
# ─────────────────────────────────────────────────────────────────────────────

class MDComparator:
    """
    Accepts a list of MDComplex objects and produces comparative plots.

    Parameters
    ----------
    complexes : list of MDComplex
        Systems to compare. compute() must have been called on each.
    n_segments : int
        Number of time segments for segmented RMSF heatmap. Default: 10.
    delta_rmsf_threshold : float
        Minimum |ΔRMSF| in Å to include in the ΔRMSF table. Default: 0 (show all).
    contact_cutoff : float
        Distance cutoff in Å for contact residue detection. Default: 5.0.
    contact_min_freq : float
        Minimum contact frequency (0–1) to report. Default: 0.30.
    """

    def __init__(
        self,
        complexes,
        n_segments=10,
        delta_rmsf_threshold=0.0,
        contact_cutoff=5.0,
        contact_min_freq=0.30,
    ):
        self.complexes = complexes
        self.n_segments = n_segments
        self.delta_rmsf_threshold = delta_rmsf_threshold
        self.contact_cutoff = contact_cutoff
        self.contact_min_freq = contact_min_freq

        # Color palette — one color per complex, scales to N
        cmap = plt.cm.get_cmap('tab10')
        self._colors = [cmap(i % 10) for i in range(len(complexes))]

        # Validation
        self._pca_valid = True
        self._delta_rmsf_valid = True
        self._run_validation()

    # ------------------------------------------------------------------
    # Validation system
    # ------------------------------------------------------------------

    def _run_validation(self):
        print('\n' + '═' * 60)
        print('MDlens Validation Report')
        print('═' * 60)

        # Check all complexes have been computed
        for cx in self.complexes:
            if not cx._computed:
                raise RuntimeError(
                    f'MDComplex [{cx.name}] has not been computed yet. '
                    f'Call cx.compute() before creating MDComparator.'
                )

        # Level 1 — Protein size mismatch
        sizes = {cx.name: cx._n_residues for cx in self.complexes}
        unique_sizes = set(sizes.values())
        if len(unique_sizes) > 1:
            print('\n❌ CRITICAL: Protein size mismatch detected.')
            for name, sz in sizes.items():
                print(f'   {name}: {sz} residues')
            print('\n   Binding site PCA and ΔRMSF comparison require the same protein target.')
            print('   These analyses will be SKIPPED.')
            print('   All other analyses (RMSD, RMSF, Rg, FEL, etc.) will still run independently.')
            self._pca_valid = False
            self._delta_rmsf_valid = False
        else:
            print(f'\n✓ Protein size: all complexes have {list(unique_sizes)[0]} residues.')

        # Level 2 — Contact residue overlap (only if PCA is valid)
        if self._pca_valid and len(self.complexes) >= 2:
            contact_sets = [
                set(cx._results.get('contact_residues', {}).keys())
                for cx in self.complexes
            ]
            # Check pairwise overlap
            for i in range(len(self.complexes)):
                for j in range(i + 1, len(self.complexes)):
                    a, b = contact_sets[i], contact_sets[j]
                    if len(a | b) == 0:
                        continue
                    overlap = len(a & b) / len(a | b)
                    if overlap < 0.80:
                        pct = int(overlap * 100)
                        print(f'\n⚠️  WARNING: Contact residue overlap is {pct}% '
                              f'between {self.complexes[i].name} and {self.complexes[j].name}.')
                        print(f'   {self.complexes[i].name} contact residues: {sorted(a)}')
                        print(f'   {self.complexes[j].name} contact residues: {sorted(b)}')
                        shared = sorted(a & b)
                        print(f'   Shared residues used for PCA: {shared}')
                        print('\n   Tip: If structures have different residue numbering, use resid_offset parameter.')
                        print('   e.g. complex2 = MDComplex(..., resid_offset=10)')

        # Level 3 — PCA advisory
        if self._pca_valid:
            print('\nℹ️  PCA NOTE: Binding site PCA comparison assumes the same protein target')
            print('    across all complexes. If comparing different proteins, results are not valid.')

        print('\n' + '═' * 60 + '\n')

    # ------------------------------------------------------------------
    # Plot helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _save_fig(fig, fname, dpi=300):
        fig.savefig(fname, dpi=dpi, bbox_inches='tight')
        print(f'  Saved: {fname}')

    def _overlay_time_plot(
        self, metric_key, ylabel, title, fname,
        hline=None, hline_label=None
    ):
        """
        Generic overlay plot for time-series metrics.
        metric_key: key in cx._results returning (time, mean, std)
        """
        fig, ax = plt.subplots(figsize=(10, 4))
        any_plotted = False
        for cx, color in zip(self.complexes, self._colors):
            data = cx._results.get(metric_key)
            if data is None:
                continue
            time, mean, std = data
            if mean is None:
                continue
            n = min(len(time), len(mean))
            ax.plot(time[:n], mean[:n], color=color, lw=1.5, label=cx.name)
            if std is not None:
                ax.fill_between(time[:n], mean[:n] - std[:n], mean[:n] + std[:n],
                                color=color, alpha=0.2)
            any_plotted = True
        if hline is not None:
            ax.axhline(hline, color='black', ls='--', lw=1.0,
                       label=hline_label or f'{hline} Å threshold')
        ax.set_xlabel('Time (ns)')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if any_plotted:
            ax.legend(framealpha=0.8)
        plt.tight_layout()
        self._save_fig(fig, fname)
        plt.show()
        plt.close(fig)

    # ------------------------------------------------------------------
    # Overlay plots
    # ------------------------------------------------------------------

    def plot_rmsd(self):
        """Overlay backbone RMSD vs time for all complexes."""
        self._overlay_time_plot(
            'rmsd', 'RMSD (Å)', 'Backbone RMSD vs Time', 'MDlens_rmsd.png'
        )

    def plot_ligand_rmsd(self):
        """Overlay ligand heavy-atom RMSD vs time for all complexes."""
        self._overlay_time_plot(
            'ligand_rmsd', 'Ligand RMSD (Å)', 'Ligand RMSD vs Time', 'MDlens_ligand_rmsd.png'
        )

    def plot_rg(self):
        """Overlay radius of gyration vs time for all complexes."""
        self._overlay_time_plot(
            'rg', 'Rg (Å)', 'Radius of Gyration vs Time', 'MDlens_rg.png'
        )

    def plot_hbonds(self):
        """Overlay protein-ligand H-bond count vs time."""
        print('\n⚠  H-BOND NOTE: Non-standard ligands may require explicit donor/acceptor hints.')
        print('   If H-bond counts appear as zero, specify donors_sel/acceptors_sel manually.')
        self._overlay_time_plot(
            'hbonds', 'H-bond Count', 'Protein–Ligand H-bonds vs Time', 'MDlens_hbonds.png'
        )

    def plot_pl_distance(self):
        """Overlay protein-ligand minimum distance vs time with 4 Å contact threshold."""
        self._overlay_time_plot(
            'pl_distance', 'Min Distance (Å)', 'Protein–Ligand Minimum Distance vs Time',
            'MDlens_pl_distance.png',
            hline=4.0, hline_label='4 Å contact threshold'
        )

    def plot_binding_energy(self):
        """Overlay simplified MM binding energy vs time."""
        self._overlay_time_plot(
            'binding_energy', 'MM Binding Energy (kJ/mol)',
            'Approximate MM Binding Energy vs Time\n(Coulomb + LJ, simplified estimate)',
            'MDlens_binding_energy.png'
        )

    def plot_rmsf(self):
        """Overlay per-residue average RMSF for all complexes."""
        fig, ax = plt.subplots(figsize=(12, 4))
        any_plotted = False
        for cx, color in zip(self.complexes, self._colors):
            resids, mean, std = cx._results.get('rmsf', (None, None, None))
            if mean is None:
                continue
            ax.plot(resids, mean, color=color, lw=1.5, label=cx.name)
            if std is not None:
                ax.fill_between(resids, mean - std, mean + std,
                                color=color, alpha=0.2)
            any_plotted = True
        ax.set_xlabel('Residue Number')
        ax.set_ylabel('RMSF (Å)')
        ax.set_title('Per-Residue RMSF (Cα)')
        if any_plotted:
            ax.legend(framealpha=0.8)
        plt.tight_layout()
        self._save_fig(fig, 'MDlens_rmsf.png')
        plt.show()
        plt.close(fig)

    def plot_all_overlay(self):
        """Run all overlay plots in sequence."""
        print('\nGenerating overlay plots...')
        for fn in [
            self.plot_rmsd,
            self.plot_rmsf,
            self.plot_rg,
            self.plot_hbonds,
            self.plot_ligand_rmsd,
            self.plot_pl_distance,
            self.plot_binding_energy,
        ]:
            try:
                fn()
            except Exception as e:
                print(f'  ✗ {fn.__name__} failed: {e}')

    # ------------------------------------------------------------------
    # Per-complex: FEL
    # ------------------------------------------------------------------

    def plot_fel(self, cx, sigma=1.0):
        """
        Free Energy Landscape using RMSD vs Rg as collective variables.
        Identifies global minimum frame and prints a detailed report.
        """
        try:
            fel_data = cx._results.get('fel_data')
            if fel_data is None:
                print(f'  ✗ FEL data not available for [{cx.name}]')
                return None

            rmsd_all, rg_all, time_all, frame_map = fel_data
            if len(rmsd_all) == 0:
                return None

            # 2D histogram
            n_bins = 50
            H, xedges, yedges = np.histogram2d(rmsd_all, rg_all, bins=n_bins)
            H = gaussian_filter(H + 1e-10, sigma=sigma)  # avoid log(0)
            G = -np.log(H / H.max())  # free energy in kT units

            xc = 0.5 * (xedges[:-1] + xedges[1:])
            yc = 0.5 * (yedges[:-1] + yedges[1:])

            # Global minimum
            min_idx = np.unravel_index(G.argmin(), G.shape)
            min_rmsd = xc[min_idx[0]]
            min_rg = yc[min_idx[1]]

            # Find closest frame
            dists_to_min = np.sqrt((rmsd_all - min_rmsd) ** 2 + (rg_all - min_rg) ** 2)
            closest_frame = int(dists_to_min.argmin())

            print(f'\n── FEL Global Minimum Report: [{cx.name}] ──')
            print(f'  Frame index : {closest_frame}')
            print(f'  Time        : {time_all[closest_frame]:.3f} ns')
            print(f'  RMSD        : {rmsd_all[closest_frame]:.3f} Å')
            print(f'  Rg          : {rg_all[closest_frame]:.3f} Å')

            cx._results['fel_min_frame'] = closest_frame
            cx._results['fel_min_rmsd'] = rmsd_all[closest_frame]
            cx._results['fel_min_rg'] = rg_all[closest_frame]

            fig, ax = plt.subplots(figsize=(7, 6))
            XX, YY = np.meshgrid(xc, yc)
            cf = ax.contourf(XX, YY, G.T, levels=30, cmap='RdYlGn_r')
            plt.colorbar(cf, ax=ax, label='Free Energy (kT)')
            ax.contour(XX, YY, G.T, levels=10, colors='black', linewidths=0.4, alpha=0.4)
            ax.scatter(
                rmsd_all[closest_frame], rg_all[closest_frame],
                marker='*', s=200, color='white', edgecolors='black',
                zorder=5, label='Global minimum'
            )
            ax.set_xlabel('Backbone RMSD (Å)')
            ax.set_ylabel('Radius of Gyration (Å)')
            ax.set_title(f'Free Energy Landscape: {cx.name}')
            ax.legend()
            plt.tight_layout()
            fname = f'MDlens_fel_{_safe_name(cx.name)}.png'
            self._save_fig(fig, fname)
            plt.show()
            plt.close(fig)
            return closest_frame

        except Exception as e:
            print(f'  ✗ FEL plot failed for [{cx.name}]: {e}')
            return None

    def plot_all_fel(self, sigma=1.0):
        """Plot FEL for every complex."""
        print('\nGenerating FEL plots...')
        for cx in self.complexes:
            try:
                self.plot_fel(cx, sigma=sigma)
            except Exception as e:
                print(f'  ✗ FEL failed for [{cx.name}]: {e}')

    # ------------------------------------------------------------------
    # Per-complex: segmented RMSF heatmap
    # ------------------------------------------------------------------

    def plot_rmsf_heatmap(self, cx):
        """
        Segmented RMSF heatmap for one complex.
        Approach inspired by eRMSF (Arantes et al., JCIM 2025, 65, 23, 12648–12654).
        """
        try:
            seg_rmsf = cx._results.get('seg_rmsf')
            if seg_rmsf is None:
                print(f'  ✗ Segmented RMSF not available for [{cx.name}]')
                return

            n_res, n_seg = seg_rmsf.shape
            resids = cx._residue_ids

            fig, ax = plt.subplots(figsize=(max(8, n_seg * 0.8), 8))
            sns.heatmap(
                seg_rmsf,
                ax=ax,
                cmap='YlOrRd',
                xticklabels=[f'Seg {i+1}' for i in range(n_seg)],
                yticklabels=False,
                cbar_kws={'label': 'RMSF (Å)'},
                linewidths=0.0,
            )
            # Tick at readable intervals (50, 100, 150...) not every residue
            tick_interval = 50 if n_res > 100 else 25 if n_res > 50 else 10
            tick_pos = list(range(0, n_res, tick_interval))
            ax.set_yticks([p + 0.5 for p in tick_pos])
            ax.set_yticklabels(
                [str(resids[p]) if p < len(resids) else '' for p in tick_pos],
                fontsize=9
            )
            ax.set_xlabel('Time Segment')
            ax.set_ylabel('Residue Number')
            ax.set_title(f'Segmented RMSF Heatmap: {cx.name}\n'
                         f'(inspired by eRMSF, Arantes et al. JCIM 2025)')
            plt.tight_layout()
            fname = f'MDlens_rmsf_heatmap_{_safe_name(cx.name)}.png'
            self._save_fig(fig, fname)
            plt.show()
            plt.close(fig)
        except Exception as e:
            print(f'  ✗ RMSF heatmap failed for [{cx.name}]: {e}')

    def plot_all_rmsf_heatmaps(self):
        """Plot segmented RMSF heatmap for every complex."""
        print('\nGenerating RMSF heatmaps...')
        for cx in self.complexes:
            self.plot_rmsf_heatmap(cx)

    def run_per_complex_analyses(self):
        """Run FEL plots and RMSF heatmaps for all complexes."""
        self.plot_all_fel()
        self.plot_all_rmsf_heatmaps()

    # ------------------------------------------------------------------
    # ΔRMSF comparative table
    # ------------------------------------------------------------------

    def compute_delta_rmsf_table(self):
        """
        Compute pairwise ΔRMSF between complexes and output a ranked table.
        For 3+ complexes, uses per-residue range (max − min).
        Saves to MDlens_delta_rmsf_table.csv.
        """
        if not self._delta_rmsf_valid:
            print('  ✗ ΔRMSF table skipped due to protein size mismatch.')
            return None

        rmsf_data = {}
        for cx in self.complexes:
            resids, mean, _ = cx._results.get('rmsf', (None, None, None))
            if mean is not None:
                rmsf_data[cx.name] = (resids, mean)

        if len(rmsf_data) < 2:
            print('  ✗ Need at least 2 complexes with RMSF data.')
            return None

        # Use first complex residue IDs as reference
        ref_name = list(rmsf_data.keys())[0]
        ref_resids = rmsf_data[ref_name][0]
        n_res = len(ref_resids)

        # Get matching residue names from first complex
        cx0 = self.complexes[0]
        res_names = cx0._residue_names if cx0._residue_names is not None else np.array(['UNK'] * n_res)

        rows = []
        all_rmsf = {}
        for name, (resids, mean) in rmsf_data.items():
            if len(mean) == n_res:
                all_rmsf[name] = mean
            else:
                all_rmsf[name] = _interpolate_to_length(mean, n_res)

        names = list(all_rmsf.keys())
        rmsf_matrix = np.array([all_rmsf[n] for n in names])  # (n_cx, n_res)

        if len(names) == 2:
            delta = rmsf_matrix[1] - rmsf_matrix[0]
            more_flex = np.where(delta > 0, names[1], names[0])
            for i in range(n_res):
                rows.append({
                    'Residue Number': int(ref_resids[i]),
                    'Residue Name': res_names[i] if i < len(res_names) else 'UNK',
                    f'RMSF {names[0]} (Å)': round(float(rmsf_matrix[0, i]), 3),
                    f'RMSF {names[1]} (Å)': round(float(rmsf_matrix[1, i]), 3),
                    'ΔRMSF (Å)': round(float(delta[i]), 3),
                    'More Flexible In': more_flex[i],
                })
        else:
            # 3+ complexes: use range
            rng = rmsf_matrix.max(axis=0) - rmsf_matrix.min(axis=0)
            most_flex = np.array(names)[rmsf_matrix.argmax(axis=0)]
            for i in range(n_res):
                row = {
                    'Residue Number': int(ref_resids[i]),
                    'Residue Name': res_names[i] if i < len(res_names) else 'UNK',
                }
                for name in names:
                    row[f'RMSF {name} (Å)'] = round(float(all_rmsf[name][i]), 3)
                row['ΔRMSF Range (Å)'] = round(float(rng[i]), 3)
                row['Most Flexible In'] = most_flex[i]
                rows.append(row)

        df = pd.DataFrame(rows)
        delta_col = 'ΔRMSF (Å)' if 'ΔRMSF (Å)' in df.columns else 'ΔRMSF Range (Å)'
        df = df[np.abs(df[delta_col]) >= self.delta_rmsf_threshold]
        df = df.sort_values(by=delta_col, key=np.abs, ascending=False).reset_index(drop=True)
        df.index = df.index + 1
        df.index.name = 'Rank'

        print('\nΔRMSF Comparative Table (Top 20):')
        print(df.head(20).to_string())
        df.to_csv('MDlens_delta_rmsf_table.csv')
        print('\n  Saved: MDlens_delta_rmsf_table.csv')
        return df

    # ------------------------------------------------------------------
    # Contact residue tables
    # ------------------------------------------------------------------

    def compute_contact_tables(self):
        """Print and save contact residue ranked tables for each complex."""
        print('\n── Contact Residue Tables ──')
        all_dfs = {}
        for cx in self.complexes:
            contacts = cx._results.get('contact_residues', {})
            if not contacts:
                print(f'  [{cx.name}] No contact residues found.')
                continue

            rows = []
            for rid, (freq, mean_dist, resname) in contacts.items():
                rows.append({
                    'Residue Number': rid,
                    'Residue Name': resname,
                    'Contact Frequency (%)': round(freq * 100, 1),
                    'Mean Min Distance (Å)': round(mean_dist, 3),
                })

            df = pd.DataFrame(rows)
            df = df.sort_values('Contact Frequency (%)', ascending=False).reset_index(drop=True)
            df.index = df.index + 1
            df.index.name = 'Rank'

            print(f'\n[{cx.name}] Contact residues:')
            print(df.to_string())

            fname = f'MDlens_contact_residues_{_safe_name(cx.name)}.csv'
            df.to_csv(fname)
            print(f'  Saved: {fname}')
            all_dfs[cx.name] = df

        return all_dfs

    # ------------------------------------------------------------------
    # Binding site PCA
    # ------------------------------------------------------------------

    def plot_binding_site_pca(self):
        """
        PCA on Cα positions of binding-site residues (union across all complexes).
        Joint PCA fit; per-complex projections overlaid on one figure.
        FEL global minimum frame for each complex is marked with a star.
        """
        if not self._pca_valid:
            print('  ✗ Binding site PCA skipped due to protein size mismatch.')
            return

        try:
            print('\nRunning Binding Site PCA...')

            # Union of contact residue IDs across all complexes
            union_resids = set()
            for cx in self.complexes:
                union_resids.update(cx._results.get('contact_residues', {}).keys())
            union_resids = sorted(union_resids)

            if len(union_resids) < 3:
                print('  ✗ Too few shared contact residues for PCA (need ≥ 3).')
                return

            print(f'  Using {len(union_resids)} binding-site residues: {union_resids}')

            # Collect Cα positions for contact residues from each complex/frame
            all_feature_rows = []  # for joint PCA fit
            per_complex_rows = {}  # complex_name → [feature_vectors]

            for cx in self.complexes:
                per_complex_rows[cx.name] = []
                for traj in cx.trajectories:
                    u = mda.Universe(cx.topology, traj)
                    protein = u.select_atoms(cx.protein_selection)
                    for ts in u.trajectory:
                        row = []
                        for rid in union_resids:
                            try:
                                res_atoms = protein.select_atoms(f'resid {rid - cx.resid_offset} and name CA')
                                if len(res_atoms) > 0:
                                    row.extend(res_atoms.positions[0])
                                else:
                                    row.extend([0.0, 0.0, 0.0])
                            except Exception:
                                row.extend([0.0, 0.0, 0.0])
                        all_feature_rows.append(row)
                        per_complex_rows[cx.name].append(row)

            X_all = np.array(all_feature_rows)
            pca = PCA(n_components=2)
            pca.fit(X_all)

            var_exp = pca.explained_variance_ratio_ * 100
            print(f'  PC1: {var_exp[0]:.1f}%  PC2: {var_exp[1]:.1f}% variance explained')

            fig, ax = plt.subplots(figsize=(8, 7))
            frame_offset = 0

            for cx, color in zip(self.complexes, self._colors):
                X_cx = np.array(per_complex_rows[cx.name])
                coords = pca.transform(X_cx)
                ax.scatter(coords[:, 0], coords[:, 1],
                           color=color, alpha=0.3, s=8, label=cx.name)
                _confidence_ellipse(
                    coords[:, 0], coords[:, 1], ax,
                    facecolor='none', edgecolor=color, lw=2.0, alpha=0.8
                )
                # Mark FEL minimum frame
                fel_min_idx = cx._results.get('fel_min_frame')
                if fel_min_idx is not None and fel_min_idx < len(X_cx):
                    min_coord = pca.transform(X_cx[fel_min_idx:fel_min_idx+1])
                    ax.scatter(
                        min_coord[0, 0], min_coord[0, 1],
                        marker='*', s=250, color=color,
                        edgecolors='black', zorder=6
                    )
                frame_offset += len(X_cx)

            ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
            ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
            ax.set_title('Binding Site PCA\n(95% confidence ellipses; ★ = FEL global minimum)')
            ax.legend(framealpha=0.9)
            plt.tight_layout()
            self._save_fig(fig, 'MDlens_binding_site_pca.png')
            plt.show()
            plt.close(fig)

        except Exception as e:
            print(f'  ✗ Binding site PCA failed: {e}')
            import traceback; traceback.print_exc()

    # ------------------------------------------------------------------
    # Combined contact + ΔRMSF run
    # ------------------------------------------------------------------

    def run_contact_and_delta_rmsf(self):
        """Run contact residue tables and ΔRMSF table together."""
        self.compute_contact_tables()
        self.compute_delta_rmsf_table()


print('✓ MDlens classes loaded successfully.')
print('  MDComplex  — defines one protein-ligand system')
print('  MDComparator — compares and plots multiple systems')

In [ ]:
#@title ### Cell 4 — Define Your Complexes
#@markdown Edit the file paths, names, and selections for your systems below.
#@markdown Then run this cell to build the MDComplex objects.

# ─────────────────────────────────────────────────────────────────────────────
# USER INPUTS — edit these
# ─────────────────────────────────────────────────────────────────────────────

complex1 = MDComplex(
    name='Compound A',
    topology='compA.pdb',                          # topology file
    trajectories=['rep1.dcd', 'rep2.dcd', 'rep3.dcd'],  # one or more replicates
    protein_selection='protein',
    ligand_selection='resname LIG',
    total_time_ns=100.0,   # <-- SET THIS: total simulation time in ns
                           # e.g. 100 for a 100 ns run, 500 for 500 ns
    resid_offset=0,
)

complex2 = MDComplex(
    name='Compound B',
    topology='compB.pdb',
    trajectories='single_rep.dcd',     # single replicate as string
    protein_selection='protein',
    ligand_selection='resname LIG',
    total_time_ns=100.0,   # <-- SET THIS: total simulation time in ns
    resid_offset=0,
)

# Add more complexes as needed:
# complex3 = MDComplex(name='Compound C', topology='compC.gro', ...)

# ─────────────────────────────────────────────────────────────────────────────
# Compute all metrics (runs on all replicates)
# ─────────────────────────────────────────────────────────────────────────────
N_SEGMENTS = 10   #@param {type:"integer"}

complex1.compute(n_segments=N_SEGMENTS)
complex2.compute(n_segments=N_SEGMENTS)
# complex3.compute(n_segments=N_SEGMENTS)

# ─────────────────────────────────────────────────────────────────────────────
# Build comparator
# ─────────────────────────────────────────────────────────────────────────────
comparator = MDComparator(
    complexes=[complex1, complex2],  # add complex3 etc. as needed
    n_segments=N_SEGMENTS,
    delta_rmsf_threshold=0.0,   # min |ΔRMSF| Å to show in table (0 = show all)
    contact_cutoff=5.0,         # Å
    contact_min_freq=0.30,      # 0–1 fraction of frames
)

print('\n✓ Comparator ready. Proceed to Cell 5 to generate plots.')

In [ ]:
#@title ### Cell 5 — Run Comparator: All Overlay Plots
#@markdown Generates all comparative overlay plots:
#@markdown RMSD, RMSF, Rg, H-bonds, Ligand RMSD, P-L Distance, Binding Energy

comparator.plot_all_overlay()

In [ ]:
#@title ### Cell 6 — Run Per-Complex Analyses (FEL, RMSF Heatmaps)
#@markdown Generates one FEL plot and one segmented RMSF heatmap per complex.
#@markdown Also prints the FEL global minimum report for each complex.

FEL_SIGMA = 1.0  #@param {type:"number"}

comparator.plot_all_fel(sigma=FEL_SIGMA)
comparator.plot_all_rmsf_heatmaps()

In [ ]:
#@title ### Cell 7 — Contact Residue Detection & ΔRMSF Table
#@markdown Computes and displays:
#@markdown - Ranked contact residue tables per complex (saved as CSV)
#@markdown - ΔRMSF comparative table (saved as CSV)

comparator.run_contact_and_delta_rmsf()

In [ ]:
#@title ### Cell 8 — Binding Site PCA
#@markdown Joint PCA on Cα positions of binding-site contact residues.
#@markdown Each point is one trajectory frame; 95% confidence ellipses drawn per complex.
#@markdown FEL global minimum frames are marked with a star (★).
#@markdown
#@markdown NOTE: Requires Cell 6 and Cell 7 to have been run first.

comparator.plot_binding_site_pca()

In [ ]:
#@title ### Cell 9 — Individual Analysis Options
#@markdown Run any single metric on demand. Uncomment the line you want.

# comparator.plot_rmsd()
# comparator.plot_rmsf()
# comparator.plot_rg()
# comparator.plot_hbonds()
# comparator.plot_ligand_rmsd()
# comparator.plot_pl_distance()
# comparator.plot_binding_energy()
# comparator.plot_fel(complex1)
# comparator.plot_fel(complex2)
# comparator.plot_rmsf_heatmap(complex1)
# comparator.plot_rmsf_heatmap(complex2)
# comparator.compute_contact_tables()
# comparator.compute_delta_rmsf_table()
# comparator.plot_binding_site_pca()

In [ ]:
#@title ### Cell 10 — Frame Extraction Utility
#@markdown Extract a PDB snapshot at a specific simulation time.
#@markdown
#@markdown Edit the parameters below and run the cell.

import MDAnalysis as mda

def extract_frame(
    topology,
    trajectory,
    time_ns,
    output_pdb,
    selection='protein or resname LIG',
    dt_output_ps=None,
):
    """
    Extract the trajectory frame closest to time_ns and save as PDB.

    Parameters
    ----------
    topology   : str  — path to topology file
    trajectory : str  — path to trajectory file
    time_ns    : float — desired time in nanoseconds
    output_pdb : str  — output PDB file name
    selection  : str  — MDAnalysis selection string for atoms to save
    """
    u = mda.Universe(topology, trajectory)
    n_frames = len(u.trajectory)
    if dt_output_ps is not None:
        frame_times_ps = [i * dt_output_ps for i in range(n_frames)]
    else:
        frame_times_ps = []
        for ts in u.trajectory:
            frame_times_ps.append(ts.time)
    target_ps = time_ns * 1000.0
    best_frame = int(min(range(n_frames), key=lambda i: abs(frame_times_ps[i] - target_ps)))
    u.trajectory[best_frame]
    actual_time_ns = frame_times_ps[best_frame] / 1000.0
    time_diff_ns = abs(actual_time_ns - time_ns)

    print(f'Frame extraction:')
    print(f'  Requested time : {time_ns:.3f} ns')
    print(f'  Closest frame  : {best_frame}')
    print(f'  Actual time    : {actual_time_ns:.3f} ns')
    print(f'  Time difference: {time_diff_ns:.3f} ns')

    atoms = u.select_atoms(selection)
    atoms.write(output_pdb)
    print(f'  Saved {len(atoms)} atoms to: {output_pdb}')


# ─────────────────────────────────────────────────────────────────────────────
# USER INPUTS — edit these
# ─────────────────────────────────────────────────────────────────────────────

TOPOLOGY   = 'compA.pdb'    #@param {type:"string"}
TRAJECTORY = 'rep1.dcd'     #@param {type:"string"}
TIME_NS    = 50.0            #@param {type:"number"}
OUTPUT_PDB = 'extracted_frame.pdb'  #@param {type:"string"}
SELECTION    = 'protein or resname LIG'  #@param {type:"string"}
DT_OUTPUT_PS = None  #@param  # Set to e.g. 10.0 if time axis is wrong (ps between saved frames)

extract_frame(TOPOLOGY, TRAJECTORY, TIME_NS, OUTPUT_PDB, SELECTION, DT_OUTPUT_PS)